# DefectosCafeVerde — train-only dual-side reliability audit

Menguji apakah bukti distribusi skor native dapat memilih sisi biji yang lebih dapat dipercaya. Audit ini hanya membaca cache train DVF1; tidak memuat validation atau test dan tidak melatih detektor.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, json

def sha256(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8*1024*1024),b''): digest.update(chunk)
    return digest.hexdigest()

roots=[Path('/content/drive/MyDrive'),Path('/content/drive/.shortcut-targets-by-id')]
summary_name='DVF1_three_seed_confirmation.json'
matches=sorted({p for root in roots if root.exists() for p in root.rglob(summary_name) if p.is_file() and p.parent.name=='defectoscafeverde-dvf1-confirmation-v1'})
if not matches: raise FileNotFoundError(f'Tidak menemukan {summary_name} di Drive bersama')
by_hash={}
for path in matches: by_hash.setdefault(sha256(path),[]).append(path)
if len(by_hash)!=1: raise RuntimeError(f'Ditemukan aggregate confirmation berbeda: {by_hash}')
aliases=next(iter(by_hash.values()))
preferred=[p for p in aliases if '/MyDrive/' in p.as_posix()]
CONFIRMATION_DECISION=(preferred or aliases)[0]
PROJECT=CONFIRMATION_DECISION.parents[2]
CONFIRMATION_ARM=CONFIRMATION_DECISION.parent/'DVF1_seed123_result.json'
TRAIN_CACHE=PROJECT/'experiments/defectoscafeverde-dvf1-v1/train_pair_cache.pt'
OUTPUT=PROJECT/'experiments/defectoscafeverde-dvsr-routability-v1/dvsr_routability.json'
for path in (CONFIRMATION_ARM,TRAIN_CACHE):
    if not path.is_file(): raise FileNotFoundError(path)
decision=json.loads(CONFIRMATION_DECISION.read_text())
assert decision['decision']=='PASS' and decision['next']=='PROMOTE_DVF1_DUAL_VIEW_FUSION'
assert decision['training_executed'] is False and decision['test_images_accessed'] is False
print('PROJECT:',PROJECT)
print('TRAIN CACHE:',TRAIN_CACHE)
print('OUTPUT:',OUTPUT)


In [ ]:
import importlib, os, shutil, subprocess, sys
WORK=Path('/content'); REPO=WORK/'coffee-bean-detection'; BRANCH='codex/defectoscafeverde-dual-view-fusion'
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_defectoscafeverde_dual_side_routability.py'],cwd=REPO,check=True)


In [ ]:
OUTPUT.parent.mkdir(parents=True,exist_ok=True)
command=[sys.executable,'-u','-m','coffee_detector.analysis.defectoscafeverde_dual_side_routability','--train-cache',str(TRAIN_CACHE),'--confirmation-arm',str(CONFIRMATION_ARM),'--confirmation-decision',str(CONFIRMATION_DECISION),'--output',str(OUTPUT)]
print('MENJALANKAN TRAIN-ONLY OOF SIDE-RELIABILITY AUDIT')
subprocess.run(command,cwd=REPO,check=True)
result=json.loads(OUTPUT.read_text())
assert result['validation_images_accessed'] is False
assert result['test_images_accessed'] is False
assert result['detector_training_executed'] is False


In [ ]:
import pandas as pd
metric_keys=['paper_accuracy','max_margin_accuracy','min_entropy_accuracy','oof_selector_accuracy','pair_oracle_accuracy','oracle_gain_over_paper','selector_gain_over_paper','captured_oracle_gain']
display(pd.DataFrame([{'metric':key,'value':result['metrics'][key]} for key in metric_keys]).style.format({'value':'{:.2%}'}))
display(pd.DataFrame.from_dict(result['folds'],orient='index'))
classwise=pd.DataFrame.from_dict(result['classwise'],orient='index').sort_values(['net','pairs'],ascending=[False,False])
display(classwise)
print('COUNTS:',{key:result['metrics'][key] for key in ('pairs','exactly_one_correct_pairs','neither_correct_pairs','both_correct_pairs')})
print('TRANSITIONS:',result['transitions'])
print('CRITERIA:',result['criteria'])
print('DECISION:',result['decision'],'| NEXT:',result['next'])
print('DETECTOR TRAINING:',result['detector_training_executed'],'| VALIDATION:',result['validation_images_accessed'],'| TEST:',result['test_images_accessed'])
print('SUMMARY:',OUTPUT)
